# Task 1 — Thematic Factor Discovery: Data Processing

Covers **Task 1, Sub-step 1** from the project spec: pre-process SEC filing HTML files.

1. **Extract sections** — Parse raw iXBRL/legacy HTML, extract MD&A and Risk Factors via TOC anchor parsing
2. **Parse subsections** — Split extracted HTML into logical subsections by bold headings (MD&A) and risk item blocks (Risk Factors)

| | |
|---|---|
| **Input** | `Data/{TICKER}/10-K/*.html`, `Data/{TICKER}/10-Q/*.html` |
| **Output (step 1)** | `output/extracted_sections/{TICKER}/{STEM}/mda.html` + `risk_factors.html` |
| **Output (step 2)** | `output/subsections/{TICKER}/{STEM}_subsections.json` |
| **GPU** | Not required — pure HTML parsing |

## Step 1: Imports & Configuration

Set up paths, logging, and regex patterns for identifying section boundaries in SEC filings.

In [ ]:
import re
import html
import json
import logging
import sys
from typing import Optional
from pathlib import Path
from dataclasses import dataclass

from bs4 import BeautifulSoup, Tag, NavigableString
from tqdm.notebook import tqdm

# ── Paths ────────────────────────────────────────────────────────────────
# Notebook lives in: fillings/roshan/Actual_code/task_1/
# Data lives in:     fillings/Data/
# Output goes to:    fillings/roshan/Actual_code/task_1/output/
_cwd = Path.cwd()

# Find the fillings/ root by walking up until we find the Data/ directory
_search = _cwd
FILLINGS_ROOT = None
for _ in range(6):
    if (_search / "Data").is_dir():
        FILLINGS_ROOT = _search
        break
    _search = _search.parent

if FILLINGS_ROOT is None:
    raise FileNotFoundError(
        f"Could not find Data/ directory in any parent of {_cwd}. "
        "Make sure the notebook is inside the fillings/ project tree."
    )

DATA_ROOT = FILLINGS_ROOT / "Data"
OUTPUT_ROOT = _cwd / "output"
EXTRACTED_SECTIONS_DIR = OUTPUT_ROOT / "extracted_sections"
SUBSECTIONS_DIR = OUTPUT_ROOT / "subsections"
TICKER_MAPPING_PATH = _cwd / "ticker_mapping.json"

EXTRACTED_SECTIONS_DIR.mkdir(parents=True, exist_ok=True)
SUBSECTIONS_DIR.mkdir(parents=True, exist_ok=True)

# ── Logging ──────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    datefmt="%H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,
)
log = logging.getLogger("task_1")

# ── Section boundary patterns ────────────────────────────────────────────
MDA_PATTERNS = [
    re.compile(r"management.{0,5}s?\s+discussion\s+and\s+analysis", re.IGNORECASE),
    re.compile(r"MD&A", re.IGNORECASE),
]
RISK_FACTORS_PATTERNS = [
    re.compile(r"risk\s+factors", re.IGNORECASE),
]

MDA_END_PATTERNS_10K = [
    re.compile(r"quantitative\s+and\s+qualitative\s+disclosures?\s+about\s+market\s+risk", re.IGNORECASE),
    re.compile(r"item\s+7a", re.IGNORECASE),
]
MDA_END_PATTERNS_10Q = [
    re.compile(r"quantitative\s+and\s+qualitative\s+disclosures?\s+about\s+market\s+risk", re.IGNORECASE),
    re.compile(r"item\s+3", re.IGNORECASE),
]
RISK_FACTORS_END_PATTERNS_10K = [
    re.compile(r"unresolved\s+staff\s+comments", re.IGNORECASE),
    re.compile(r"item\s+1b", re.IGNORECASE),
    re.compile(r"item\s+1c", re.IGNORECASE),
]
RISK_FACTORS_END_PATTERNS_10Q = [
    re.compile(r"management.{0,5}s?\s+discussion\s+and\s+analysis", re.IGNORECASE),
    re.compile(r"item\s+2", re.IGNORECASE),
]

# Rough token estimate: ~4 chars per token
CHARS_PER_TOKEN = 4

print(f"Fillings root:   {FILLINGS_ROOT}")
print(f"Data root:       {DATA_ROOT}")
print(f"Sections dir:    {EXTRACTED_SECTIONS_DIR}")
print(f"Subsections dir: {SUBSECTIONS_DIR}")

## Step 2: HTML Parsing Utilities

Core functions for parsing SEC filing HTML:
- **TOC parsing** — extract anchor-ID-to-title mapping from the Table of Contents
- **Anchor finding** — locate section start/end positions in the raw HTML (handles both `id=` for iXBRL and `name=` for legacy)
- **HTML cleaning** — strip iXBRL tags, styles, scripts; convert to minimal HTML with `<b>` headings, `<p>` paragraphs, simplified `<table>`s

In [ ]:
@dataclass
class SectionResult:
    """Result of extracting a section from a filing."""
    section_name: str
    html_content: str
    text_length: int
    success: bool
    error: Optional[str] = None


def detect_form_type(filepath: Path) -> str:
    """Detect 10-K or 10-Q from the filename."""
    name = filepath.name.upper()
    if "10-K" in name:
        return "10-K"
    elif "10-Q" in name:
        return "10-Q"
    raise ValueError(f"Cannot detect form type from filename: {filepath.name}")


def detect_format(raw_html: str) -> str:
    """Detect whether filing is iXBRL or legacy HTML."""
    if "xmlns:ix=" in raw_html[:5000] or "<ix:" in raw_html[:10000]:
        return "ixbrl"
    return "legacy"


def parse_toc(soup: BeautifulSoup, raw_html: str) -> dict[str, str]:
    """
    Parse the Table of Contents to build anchor_id -> section_title mapping.
    Scans the first 20% of the document for <a href=\"#...\"> links.
    """
    toc_links: dict[str, list[str]] = {}
    cutoff = len(raw_html) // 5
    toc_soup = BeautifulSoup(raw_html[:cutoff], "lxml")

    for a_tag in toc_soup.find_all("a", href=True):
        href = a_tag["href"]
        if not href.startswith("#"):
            continue
        anchor_id = href[1:]
        text = a_tag.get_text(strip=True)
        if not text or text.isdigit() or "table of contents" in text.lower():
            continue
        if anchor_id not in toc_links:
            toc_links[anchor_id] = []
        toc_links[anchor_id].append(text)

    toc_map: dict[str, str] = {}
    for anchor_id, texts in toc_links.items():
        combined = html.unescape(" ".join(texts))
        toc_map[anchor_id] = combined
    return toc_map


def find_section_anchor(
    toc_map: dict[str, str], patterns: list[re.Pattern]
) -> Optional[str]:
    """Find the anchor ID for a section by matching TOC titles against patterns."""
    for anchor_id, title in toc_map.items():
        for pat in patterns:
            if pat.search(title):
                return anchor_id
    return None


def find_anchor_position(raw_html: str, anchor_id: str, html_format: str) -> Optional[int]:
    """
    Find the byte position of an anchor in the raw HTML.
    Handles id=\"...\" (iXBRL) and name=\"...\" (legacy), case-insensitive.
    """
    patterns = [
        re.compile(re.escape(f'name="{anchor_id}"'), re.IGNORECASE),
        re.compile(re.escape(f"name='{anchor_id}'"), re.IGNORECASE),
        re.compile(re.escape(f'id="{anchor_id}"'), re.IGNORECASE),
        re.compile(re.escape(f"id='{anchor_id}'"), re.IGNORECASE),
    ]
    for pat in patterns:
        m = pat.search(raw_html)
        if m:
            return m.start()
    return None


def find_end_boundary(
    raw_html: str,
    start_pos: int,
    toc_map: dict[str, str],
    end_patterns: list[re.Pattern],
    html_format: str,
) -> int:
    """
    Find the end boundary of a section.
    Strategy: TOC anchor -> heading text search -> 500KB fallback.
    """
    # Strategy 1: next section via TOC
    next_anchor = find_section_anchor(toc_map, end_patterns)
    if next_anchor:
        pos = find_anchor_position(raw_html, next_anchor, html_format)
        if pos and pos > start_pos:
            return pos

    # Strategy 2: heading text search after start_pos
    search_region = raw_html[start_pos:]
    for pat in end_patterns:
        matches = list(pat.finditer(search_region))
        for m in matches:
            if m.start() > 500:
                return start_pos + m.start()

    # Strategy 3: 500KB fallback
    return min(start_pos + 500_000, len(raw_html))


# ── HTML cleaning helpers ─────────────────────────────────────────────────

def _is_bold(tag: Tag) -> bool:
    if tag.name in ("b", "strong"):
        return True
    style = tag.get("style", "")
    if "font-weight:700" in style or "font-weight:bold" in style or "font-weight: bold" in style:
        return True
    if tag.name == "font":
        style = tag.get("style", "")
        if "font-weight:bold" in style or "font-weight:700" in style:
            return True
    return False


def _is_italic(tag: Tag) -> bool:
    if tag.name in ("i", "em"):
        return True
    style = tag.get("style", "")
    return "font-style:italic" in style


def _get_text_content(tag) -> str:
    text = tag.get_text(separator=" ", strip=False)
    text = html.unescape(text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)
    return text.strip()


def _convert_table(table_tag: Tag) -> Optional[str]:
    """Convert a table to simplified HTML, preserving structure."""
    rows = table_tag.find_all("tr")
    if not rows:
        return None
    all_text = _get_text_content(table_tag)
    if len(all_text.strip()) < 20:
        return None

    out = ["<table>"]
    for row in rows:
        cells = row.find_all(["td", "th"])
        if not cells:
            continue
        cell_texts = []
        for cell in cells:
            text = _get_text_content(cell)
            tag_name = "th" if cell.name == "th" or _is_bold(cell) else "td"
            cell_texts.append(f"<{tag_name}>{text}</{tag_name}>")
        if any(_get_text_content(c) for c in cells):
            out.append(f"<tr>{''.join(cell_texts)}</tr>")
    out.append("</table>")
    if len(out) <= 2:
        return None
    return "\n".join(out)


def _walk_and_convert(element, parts: list[str], depth: int = 0) -> None:
    """
    Recursively walk the DOM and convert to minimal HTML.
    Bold short text -> <b>, regular paragraphs -> <p>, tables simplified.
    """
    if isinstance(element, NavigableString):
        return
    if not isinstance(element, Tag):
        return

    tag = element
    style = tag.get("style", "")
    if "display:none" in style or "display: none" in style:
        return

    if tag.name == "table":
        table_html = _convert_table(tag)
        if table_html:
            parts.append(table_html)
        return

    if tag.name == "hr":
        parts.append("<hr>")
        return

    block_tags = {"div", "p", "section", "article", "blockquote", "li", "td", "th"}

    if tag.name in block_tags:
        text = _get_text_content(tag)
        if not text:
            for child in tag.children:
                _walk_and_convert(child, parts, depth + 1)
            return

        is_heading = False
        if len(text) < 200:
            if _is_bold(tag):
                is_heading = True
            else:
                bold_children = tag.find_all(
                    lambda t: isinstance(t, Tag) and _is_bold(t)
                )
                if bold_children:
                    bold_text = " ".join(_get_text_content(b) for b in bold_children).strip()
                    if bold_text and len(bold_text) > len(text) * 0.7:
                        is_heading = True

        if is_heading and text:
            italic = _is_italic(tag) or any(
                _is_italic(c) for c in tag.find_all(True) if isinstance(c, Tag)
            )
            if italic:
                parts.append(f"<b><i>{text}</i></b>")
            else:
                parts.append(f"<b>{text}</b>")
        elif text:
            parts.append(f"<p>{text}</p>")
        return

    for child in tag.children:
        _walk_and_convert(child, parts, depth + 1)


def clean_section_html(section_html: str) -> str:
    """Clean extracted section HTML into minimal HTML with <b> headings and <p> text."""
    soup = BeautifulSoup(section_html, "lxml")

    for el in soup.find_all(style=re.compile(r"display\s*:\s*none")):
        el.decompose()
    for tag_name in ["script", "style", "img", "link", "meta"]:
        for el in soup.find_all(tag_name):
            el.decompose()
    for el in soup.find_all(re.compile(r"^ix:")):
        el.unwrap()
    for el in soup.find_all(re.compile(r":")):
        if el.name and ":" in el.name:
            el.unwrap()

    output_parts: list[str] = []
    _walk_and_convert(soup.body or soup, output_parts)

    result = "\n".join(output_parts)
    result = re.sub(r"\n{3,}", "\n\n", result)
    return result.strip()


def _fallback_find_section_start(
    raw_html: str, section_name: str, form_type: str
) -> Optional[int]:
    """Fallback: find a section start by searching for the heading directly."""
    if section_name == "mda":
        if form_type == "10-K":
            patterns = [re.compile(
                r"item\s*7\.?\s*[-\u2013\u2014.]?\s*management.{0,5}s?\s+discussion",
                re.IGNORECASE,
            )]
        else:
            patterns = [re.compile(
                r"item\s*2\.?\s*[-\u2013\u2014.]?\s*management.{0,5}s?\s+discussion",
                re.IGNORECASE,
            )]
    elif section_name == "risk_factors":
        patterns = [re.compile(
            r"item\s*1a\.?\s*[-\u2013\u2014.]?\s*risk\s+factors",
            re.IGNORECASE,
        )]
    else:
        return None

    search_start = len(raw_html) // 10
    search_region = raw_html[search_start:]
    for pat in patterns:
        m = pat.search(search_region)
        if m:
            return search_start + m.start()
    return None


def extract_sections(filepath: Path) -> dict[str, SectionResult]:
    """
    Extract MD&A and Risk Factors from a SEC filing HTML.
    Returns dict with keys 'mda' and 'risk_factors', each a SectionResult.
    """
    form_type = detect_form_type(filepath)

    with open(filepath, "r", encoding="utf-8", errors="replace") as f:
        raw_html = f.read()

    html_format = detect_format(raw_html)
    toc_map = parse_toc(
        BeautifulSoup(raw_html[: len(raw_html) // 5], "lxml"), raw_html
    )

    if form_type == "10-K":
        sections_cfg = {
            "mda": {"start": MDA_PATTERNS, "end": MDA_END_PATTERNS_10K},
            "risk_factors": {"start": RISK_FACTORS_PATTERNS, "end": RISK_FACTORS_END_PATTERNS_10K},
        }
    else:
        sections_cfg = {
            "mda": {"start": MDA_PATTERNS, "end": MDA_END_PATTERNS_10Q},
            "risk_factors": {"start": RISK_FACTORS_PATTERNS, "end": RISK_FACTORS_END_PATTERNS_10Q},
        }

    results: dict[str, SectionResult] = {}

    for section_name, cfg in sections_cfg.items():
        try:
            anchor_id = find_section_anchor(toc_map, cfg["start"])

            if not anchor_id:
                start_pos = _fallback_find_section_start(raw_html, section_name, form_type)
                if start_pos is None:
                    results[section_name] = SectionResult(
                        section_name=section_name, html_content="",
                        text_length=0, success=False,
                        error=f"Could not find {section_name} in TOC or document",
                    )
                    continue
            else:
                start_pos = find_anchor_position(raw_html, anchor_id, html_format)
                if start_pos is None:
                    # TOC anchor exists but not found in body — try fallback
                    start_pos = _fallback_find_section_start(raw_html, section_name, form_type)
                    if start_pos is None:
                        results[section_name] = SectionResult(
                            section_name=section_name, html_content="",
                            text_length=0, success=False,
                            error=f"Anchor {anchor_id} not found in document, fallback also failed",
                        )
                        continue

            end_pos = find_end_boundary(
                raw_html, start_pos, toc_map, cfg["end"], html_format
            )

            section_html = raw_html[start_pos:end_pos]
            clean_html = clean_section_html(section_html)

            text_only = re.sub(r"<[^>]+>", "", clean_html)
            text_length = len(text_only.strip())

            if text_length < 100:
                results[section_name] = SectionResult(
                    section_name=section_name, html_content=clean_html,
                    text_length=text_length, success=False,
                    error=f"Extracted section too short ({text_length} chars)",
                )
            else:
                results[section_name] = SectionResult(
                    section_name=section_name, html_content=clean_html,
                    text_length=text_length, success=True,
                )

        except Exception as e:
            results[section_name] = SectionResult(
                section_name=section_name, html_content="",
                text_length=0, success=False, error=str(e),
            )

    return results


print("Parser utilities loaded.")

## Step 3: Discover Filings

Load tickers from `ticker_mapping.json`, find all filing HTML files, and display a dry-run summary of what will be processed.

In [ ]:
# Load tickers
with open(TICKER_MAPPING_PATH) as f:
    ticker_mapping = json.load(f)

all_tickers = sorted(
    t for sector_list in ticker_mapping.values() for t in sector_list
)
print(f"Loaded {len(all_tickers)} tickers across {len(ticker_mapping)} sub-sectors\n")
for sector, tickers in ticker_mapping.items():
    print(f"  {sector}: {len(tickers)} tickers")


def find_filings(ticker: str) -> list[Path]:
    """Find all filing HTML files for a ticker."""
    ticker_dir = DATA_ROOT / ticker
    if not ticker_dir.exists():
        return []
    filings = []
    for form_dir in ["10-K", "10-Q"]:
        form_path = ticker_dir / form_dir
        if form_path.exists():
            filings.extend(sorted(form_path.glob("*.html")))
    return filings


def output_path_for_filing(filepath: Path) -> Path:
    """Compute output directory for a filing."""
    stem = filepath.stem
    ticker = stem.split("_")[0]
    return EXTRACTED_SECTIONS_DIR / ticker / stem


def is_already_extracted(output_dir: Path) -> bool:
    """Check if mda.html already exists (resume-safe)."""
    return (output_dir / "mda.html").exists()


# ── Dry-run summary ───────────────────────────────────────────────────────
total_filings = 0
total_already = 0
tickers_with_filings = 0
tickers_missing = []

for ticker in all_tickers:
    filings = find_filings(ticker)
    if filings:
        tickers_with_filings += 1
        total_filings += len(filings)
        total_already += sum(
            1 for f in filings if is_already_extracted(output_path_for_filing(f))
        )
    else:
        tickers_missing.append(ticker)

print(f"\n{tickers_with_filings}/{len(all_tickers)} tickers have filings")
print(f"{total_filings} total filing HTMLs")
print(f"{total_already} already extracted (will be skipped)")
print(f"{total_filings - total_already} remaining to process")

if tickers_missing:
    print(f"\nMissing tickers ({len(tickers_missing)}): {', '.join(tickers_missing)}")

## Step 4: Run Extraction

Process each filing: parse TOC, find MD&A and Risk Factors anchors, extract and clean the HTML.

**Resume-safe** — filings with existing `mda.html` output are skipped automatically.

In [ ]:
def process_filing(filepath: Path) -> dict:
    """Process a single filing: extract MD&A and Risk Factors."""
    stem = filepath.stem
    output_dir = output_path_for_filing(filepath)

    if is_already_extracted(output_dir):
        return {"file": stem, "status": "skipped"}

    try:
        results = extract_sections(filepath)
    except Exception as e:
        log.error(f"  CRASH {stem}: {e}")
        return {"file": stem, "status": "error", "reason": str(e)}

    output_dir.mkdir(parents=True, exist_ok=True)
    status = {"file": stem, "status": "ok", "sections": {}}

    for section_name, result in results.items():
        out_file = output_dir / f"{section_name}.html"
        if result.success:
            out_file.write_text(result.html_content, encoding="utf-8")
            status["sections"][section_name] = {
                "success": True, "text_length": result.text_length,
            }
        else:
            out_file.write_text(
                f"<!-- EXTRACTION FAILED: {result.error} -->", encoding="utf-8"
            )
            status["sections"][section_name] = {
                "success": False, "error": result.error,
            }
    return status


# ── Build flat list of all filing HTMLs for progress tracking ─────────────
all_filing_paths: list[tuple[str, Path]] = []
for ticker in all_tickers:
    for filepath in find_filings(ticker):
        all_filing_paths.append((ticker, filepath))

print(f"{len(all_filing_paths)} filing jobs queued")

# ── Main extraction loop ──────────────────────────────────────────────────
all_stats: dict[str, dict] = {}
g_processed = g_skipped = g_errors = g_mda_ok = g_rf_ok = 0

pbar = tqdm(all_filing_paths, desc="Extracting sections", unit="filing")
for ticker, filepath in pbar:
    if ticker not in all_stats:
        all_stats[ticker] = {
            "ticker": ticker, "total": 0, "processed": 0,
            "skipped": 0, "errors": 0, "mda_ok": 0, "rf_ok": 0,
        }
    all_stats[ticker]["total"] += 1

    result = process_filing(filepath)
    if result["status"] == "skipped":
        g_skipped += 1
        all_stats[ticker]["skipped"] += 1
    elif result["status"] == "error":
        g_errors += 1
        all_stats[ticker]["errors"] += 1
    else:
        g_processed += 1
        all_stats[ticker]["processed"] += 1
        sections = result.get("sections", {})
        if sections.get("mda", {}).get("success"):
            g_mda_ok += 1
            all_stats[ticker]["mda_ok"] += 1
        if sections.get("risk_factors", {}).get("success"):
            g_rf_ok += 1
            all_stats[ticker]["rf_ok"] += 1

    pbar.set_postfix(ok=g_processed, skip=g_skipped, err=g_errors, mda=g_mda_ok, rf=g_rf_ok)

# Add tickers with no filings
for ticker in all_tickers:
    if ticker not in all_stats:
        all_stats[ticker] = {
            "ticker": ticker, "total": 0, "processed": 0,
            "skipped": 0, "errors": 0, "mda_ok": 0, "rf_ok": 0,
        }

all_stats = list(all_stats.values())

# Save summary
summary_path = EXTRACTED_SECTIONS_DIR / "extraction_summary.json"
with open(summary_path, "w") as f:
    json.dump({"tickers": all_stats}, f, indent=2)

print(f"\nDone. {g_processed} processed, {g_skipped} skipped, {g_errors} errors")
print(f"MD&A: {g_mda_ok} ok | Risk Factors: {g_rf_ok} ok")
print(f"Summary written to {summary_path}")

## Step 5: Extraction Summary

Review section extraction results across all tickers. Check MD&A and Risk Factors success rates and identify any failures.

In [ ]:
total_filings = sum(s["total"] for s in all_stats)
total_processed = sum(s["processed"] for s in all_stats)
total_skipped = sum(s["skipped"] for s in all_stats)
total_errors = sum(s["errors"] for s in all_stats)
total_mda = sum(s["mda_ok"] for s in all_stats)
total_rf = sum(s["rf_ok"] for s in all_stats)

print("=" * 60)
print(f"DONE: {total_filings} filings across {len(all_tickers)} tickers")
print(f"  Processed: {total_processed}")
print(f"  Skipped:   {total_skipped}")
print(f"  Errors:    {total_errors}")
print(f"  MD&A OK:   {total_mda}")
print(f"  Risk OK:   {total_rf}")
if total_processed > 0:
    print(f"  MD&A rate: {total_mda / total_processed:.1%}")
    print(f"  RF rate:   {total_rf / total_processed:.1%}")

# Show tickers with MD&A failures
failed = [s for s in all_stats if s["processed"] > 0 and s["mda_ok"] < s["processed"]]
if failed:
    print(f"\nTickers with MD&A failures ({len(failed)}):")
    for s in sorted(failed, key=lambda x: x["processed"] - x["mda_ok"], reverse=True):
        miss = s["processed"] - s["mda_ok"]
        print(f"  {s['ticker']}: {miss}/{s['processed']} MD&A failed")

# Show tickers with no filings
no_data = [s for s in all_stats if s["total"] == 0]
if no_data:
    print(f"\nTickers with no filings ({len(no_data)}): {', '.join(s['ticker'] for s in no_data)}")

## Step 6: Subsection Parsing Utilities

Split extracted HTML into logical subsections:
- **MD&A** — split by `<b>` headings. Each bold heading starts a new subsection. Very small subsections (< 50 tokens) are merged into the next one.
- **Risk Factors** — split by risk item headings (bold text that looks like a risk description: long sentences, contains "could", "may", "adversely", etc.)

Each subsection gets a plain-text body and a rough token estimate (~4 chars/token).

In [ ]:
def estimate_tokens(text: str) -> int:
    return len(text) // CHARS_PER_TOKEN


def strip_html(text: str) -> str:
    """Remove HTML tags, returning plain text."""
    return re.sub(r"<[^>]+>", "", text).strip()


def parse_mda_subsections(html_content: str) -> list[dict]:
    """
    Split MD&A HTML into subsections based on bold headings.
    Everything between two <b>...</b> headings is one subsection.
    """
    parts = re.split(r"(<b>.*?</b>)", html_content, flags=re.DOTALL)

    subsections = []
    current_name = "Introduction"
    current_text_parts: list[str] = []

    for part in parts:
        part = part.strip()
        if not part:
            continue

        bold_match = re.match(r"<b>(.*?)</b>", part, re.DOTALL)
        if bold_match:
            heading_text = strip_html(bold_match.group(1)).strip()
            if not heading_text or len(heading_text) < 3:
                continue
            if heading_text.lower() in ("table of contents",):
                continue

            # Save previous subsection if it has content
            if current_text_parts:
                text = "\n".join(current_text_parts).strip()
                plain = strip_html(text)
                if len(plain) > 50:
                    subsections.append({
                        "name": current_name,
                        "section": "mda",
                        "text": plain,
                        "token_estimate": estimate_tokens(plain),
                    })

            current_name = heading_text
            current_text_parts = []
        else:
            current_text_parts.append(part)

    # Last subsection
    if current_text_parts:
        text = "\n".join(current_text_parts).strip()
        plain = strip_html(text)
        if len(plain) > 50:
            subsections.append({
                "name": current_name,
                "section": "mda",
                "text": plain,
                "token_estimate": estimate_tokens(plain),
            })

    # Merge very small subsections (< 50 tokens) into the next one
    merged = []
    i = 0
    while i < len(subsections):
        sub = subsections[i]
        if sub["token_estimate"] < 50 and i + 1 < len(subsections):
            nxt = subsections[i + 1]
            nxt["text"] = sub["text"] + "\n\n" + nxt["text"]
            nxt["name"] = sub["name"] + " / " + nxt["name"]
            nxt["token_estimate"] = estimate_tokens(nxt["text"])
            i += 1
        else:
            merged.append(sub)
            i += 1
    return merged


def parse_risk_factors_subsections(html_content: str) -> list[dict]:
    """
    Split Risk Factors HTML into individual risk items.
    Risk factors are separated by bold headings describing each risk.
    """
    parts = re.split(r"(<b>.*?</b>)", html_content, flags=re.DOTALL)

    subsections = []
    current_name = "Risk Factors Overview"
    current_text_parts: list[str] = []

    for part in parts:
        part = part.strip()
        if not part:
            continue

        bold_match = re.match(r"<b>(.*?)</b>", part, re.DOTALL)
        if bold_match:
            heading_text = strip_html(bold_match.group(1)).strip()
            if not heading_text or len(heading_text) < 3:
                continue
            if heading_text.lower() in ("table of contents",):
                continue

            # Check if this looks like a risk factor heading
            is_risk_heading = (
                len(heading_text) > 20
                or heading_text.lower().startswith("risk")
                or "could" in heading_text.lower()
                or "may" in heading_text.lower()
                or "adversely" in heading_text.lower()
            )
            is_category = (
                heading_text.lower().startswith("risks related")
                or heading_text.lower().startswith("risk factor")
                or "item 1a" in heading_text.lower()
            )

            if is_risk_heading or is_category:
                # Save previous subsection
                if current_text_parts:
                    text = "\n".join(current_text_parts).strip()
                    plain = strip_html(text)
                    if len(plain) > 30:
                        subsections.append({
                            "name": current_name,
                            "section": "risk_factors",
                            "text": plain,
                            "token_estimate": estimate_tokens(plain),
                        })
                current_name = heading_text
                current_text_parts = []
            else:
                current_text_parts.append(heading_text)
        else:
            current_text_parts.append(part)

    # Last subsection
    if current_text_parts:
        text = "\n".join(current_text_parts).strip()
        plain = strip_html(text)
        if len(plain) > 30:
            subsections.append({
                "name": current_name,
                "section": "risk_factors",
                "text": plain,
                "token_estimate": estimate_tokens(plain),
            })

    # If we couldn't split into individual risks, return the whole thing as one block
    if len(subsections) <= 1:
        plain = strip_html(html_content)
        if len(plain) > 50:
            return [{
                "name": "Risk Factors",
                "section": "risk_factors",
                "text": plain,
                "token_estimate": estimate_tokens(plain),
            }]

    return subsections


print("Subsection parsing utilities loaded.")

## Step 7: Run Subsection Parsing

Process each filing's extracted HTML (from Step 4) into subsection JSONs.

**Resume-safe** — filings with existing `*_subsections.json` output are skipped automatically.

In [ ]:
def process_filing_subsections(filing_dir: Path) -> dict | None:
    """
    Process one filing directory containing mda.html and risk_factors.html.
    Returns subsections dict or None if nothing to process.
    """
    mda_file = filing_dir / "mda.html"
    rf_file = filing_dir / "risk_factors.html"

    # Parse filing metadata from directory name: TICKER_FORM_DATE
    parts = filing_dir.name.split("_")
    if len(parts) < 3:
        return None
    ticker = parts[0]
    form = parts[1]
    date = "_".join(parts[2:])

    all_subsections = []

    # Parse MD&A
    if mda_file.exists():
        content = mda_file.read_text(encoding="utf-8")
        if not content.startswith("<!-- EXTRACTION FAILED"):
            subs = parse_mda_subsections(content)
            all_subsections.extend(subs)

    # Parse Risk Factors
    if rf_file.exists():
        content = rf_file.read_text(encoding="utf-8")
        if not content.startswith("<!-- EXTRACTION FAILED"):
            subs = parse_risk_factors_subsections(content)
            all_subsections.extend(subs)

    if not all_subsections:
        return None

    return {
        "ticker": ticker,
        "form": form,
        "filing_date": date,
        "num_subsections": len(all_subsections),
        "total_tokens": sum(s["token_estimate"] for s in all_subsections),
        "subsections": all_subsections,
    }


# ── Build flat list of all filing dirs for progress tracking ──────────────
all_filing_dirs: list[tuple[str, Path]] = []
for ticker in all_tickers:
    ticker_input = EXTRACTED_SECTIONS_DIR / ticker
    if not ticker_input.exists():
        continue
    for d in sorted(d for d in ticker_input.iterdir() if d.is_dir()):
        all_filing_dirs.append((ticker, d))

print(f"{len(all_filing_dirs)} filing dirs queued for subsection parsing")

# ── Main subsection parsing loop ─────────────────────────────────────────
sub_stats: dict[str, dict] = {}
g_sub_processed = g_sub_skipped = g_sub_errors = g_total_subs = 0

pbar = tqdm(all_filing_dirs, desc="Parsing subsections", unit="filing")
for ticker, filing_dir in pbar:
    if ticker not in sub_stats:
        sub_stats[ticker] = {
            "ticker": ticker, "total": 0, "processed": 0,
            "skipped": 0, "errors": 0, "subsections": 0,
        }
    sub_stats[ticker]["total"] += 1

    ticker_output = SUBSECTIONS_DIR / ticker
    ticker_output.mkdir(parents=True, exist_ok=True)

    out_file = ticker_output / f"{filing_dir.name}_subsections.json"

    # Resume-safe
    if out_file.exists():
        g_sub_skipped += 1
        sub_stats[ticker]["skipped"] += 1
        pbar.set_postfix(ok=g_sub_processed, skip=g_sub_skipped, err=g_sub_errors, subs=g_total_subs)
        continue

    try:
        result = process_filing_subsections(filing_dir)
        if result:
            with open(out_file, "w") as f:
                json.dump(result, f, indent=2)
            g_sub_processed += 1
            g_total_subs += result["num_subsections"]
            sub_stats[ticker]["processed"] += 1
            sub_stats[ticker]["subsections"] += result["num_subsections"]
        else:
            g_sub_errors += 1
            sub_stats[ticker]["errors"] += 1
    except Exception as e:
        g_sub_errors += 1
        sub_stats[ticker]["errors"] += 1
        log.error(f"  {filing_dir.name}: {e}")

    pbar.set_postfix(ok=g_sub_processed, skip=g_sub_skipped, err=g_sub_errors, subs=g_total_subs)

# Add tickers with no extracted sections
for ticker in all_tickers:
    if ticker not in sub_stats:
        sub_stats[ticker] = {
            "ticker": ticker, "total": 0, "processed": 0,
            "skipped": 0, "errors": 0, "subsections": 0,
        }

sub_stats = list(sub_stats.values())

print(f"\nDone. {g_sub_processed} processed, {g_sub_skipped} skipped, {g_sub_errors} errors")
print(f"Total subsections created: {g_total_subs}")
if g_sub_processed > 0:
    print(f"Avg subsections/filing: {g_total_subs / g_sub_processed:.1f}")

## Step 8: Subsection Parsing Summary

Review subsection parsing results. Check how many subsections were extracted per filing and total token counts.

In [ ]:
total_filings_sub = sum(s["total"] for s in sub_stats)
total_processed_sub = sum(s["processed"] for s in sub_stats)
total_skipped_sub = sum(s["skipped"] for s in sub_stats)
total_errors_sub = sum(s["errors"] for s in sub_stats)
total_subsections = sum(s.get("subsections", 0) for s in sub_stats)

print("=" * 60)
print(f"SUBSECTION PARSING: {total_filings_sub} filing dirs across {len(all_tickers)} tickers")
print(f"  Processed: {total_processed_sub}")
print(f"  Skipped:   {total_skipped_sub}")
print(f"  Errors:    {total_errors_sub}")
print(f"  Total subsections created: {total_subsections}")
if total_processed_sub > 0:
    print(f"  Avg subsections/filing:    {total_subsections / total_processed_sub:.1f}")

# Show tickers with errors
failed_sub = [s for s in sub_stats if s["errors"] > 0]
if failed_sub:
    print(f"\nTickers with subsection errors ({len(failed_sub)}):")
    for s in sorted(failed_sub, key=lambda x: x["errors"], reverse=True):
        print(f"  {s['ticker']}: {s['errors']}/{s['total']} failed")

# Show tickers with no extracted sections to parse
no_input = [s for s in sub_stats if s["total"] == 0]
if no_input:
    print(f"\nTickers with no extracted sections ({len(no_input)}): "
          f"{', '.join(s['ticker'] for s in no_input)}")

# Quick peek at one subsection file
sample_dirs = list(SUBSECTIONS_DIR.iterdir())
if sample_dirs:
    sample_ticker_dir = sorted(sample_dirs)[0]
    sample_files = sorted(sample_ticker_dir.glob("*_subsections.json"))
    if sample_files:
        with open(sample_files[0]) as f:
            sample = json.load(f)
        print(f"\n── Sample: {sample_files[0].name} ──")
        print(f"  Ticker: {sample['ticker']}, Form: {sample['form']}, Date: {sample['filing_date']}")
        print(f"  Subsections: {sample['num_subsections']}, Tokens: {sample['total_tokens']}")
        for s in sample["subsections"][:5]:
            print(f"    [{s['section']}] {s['name'][:60]}... ({s['token_estimate']} tokens)")